<a href="https://colab.research.google.com/github/jpdeleon/quicklook/blob/main/notebook/ttv_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transit-timing variation (TTV) search

`ttv_search=True` fits every transit of a periodic signal on its own, with its mid-time free, and tests the observed-minus-calculated (O−C) times against a linear ephemeris. It needs at least three transits, so stitch sectors for long periods.

Method and outputs: [docs/ttv-search.md](https://github.com/jpdeleon/quicklook/blob/main/docs/ttv-search.md).

## Setup

On Google Colab, the next cell installs `quicklook` from GitHub. It takes a minute or two the first time, and no runtime restart is needed. Elsewhere it only prints the install command, so it never replaces a local install.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !python -m pip install -q "quicklook-package @ git+https://github.com/jpdeleon/quicklook.git"
else:
    print('Not on Colab. If needed, install with: pip install "quicklook-package @ git+https://github.com/jpdeleon/quicklook.git"')

In [ ]:
from importlib.metadata import version

print("quicklook-package", version("quicklook-package"))

In [ ]:
%matplotlib inline
from pathlib import Path

from IPython.display import Image, display

from quicklook import TessQuickLook

OUTDIR = Path("outputs")  # figures and *_tls.h5 files are written here
OUTDIR.mkdir(exist_ok=True)

## Example 1: TOI-216 c, sectors 1–6

TOI-216 has two planets near a 2:1 resonance with known TTVs. TLS finds TOI-216 c at P ≈ 34.55 d, all 5 transits are fitted, and the O−C swings by about ±10 min, so TTVs are flagged. Panels 7 and 8 become the transits aligned on their fitted times and the O−C diagram. This run takes several minutes.

In [ ]:
toi216 = TessQuickLook(
    "TOI-216",
    sector=[1, 2, 3, 4, 5, 6],
    exptime=120,
    window_length=1,
    ttv_search=True,
    savefig=True,
    outdir=str(OUTDIR),
    overwrite=True,
)
fig = toi216.plot_tql()

In [ ]:
s = toi216.ttv_stats
MIN_PER_DAY = 1440
print(f"{s.n} transits, O-C amplitude {s.amplitude * MIN_PER_DAY:.1f} min, "
      f"p = {s.pvalue:.1e} (without the worst transit: {s.pvalue_robust:.1e}), "
      f"TTVs detected: {s.detected}")
for epoch, oc, err in zip(s.epochs, s.oc, s.oc_err):
    print(f"  epoch {int(epoch):3d}: O-C = {oc * MIN_PER_DAY:+6.1f} ± {err * MIN_PER_DAY:.1f} min")

## Example 2: an active star, TOI-6109 b

On spotted stars TLS can lock onto the rotation signal (TOI-6109 is a ~3 d rotator with up to 40 ppt spots, and TLS peaks at 18.4 d). A `custom_ephem` `[Tc, Tc_err, P, P_err, T14, T14_err]` (Tc in BJD or BTJD, T14 in hours) takes priority: each transit of the known planet is then timed against a local baseline on the raw flux. This also writes `*_ttv.png` (per-transit and per-visit O−C) and `*_ttv_transits.png` (every transit with its fit). Expect the sector 58 visit to come out about 2.5 h late (mean O−C ≈ +160 min) and the sector 85 visit on time.

In [ ]:
toi6109 = TessQuickLook(
    "TOI-6109",
    sector=[18, 58, 85],
    exptime=120,
    ttv_search=True,
    custom_ephem=[2458791.0538, 0.001, 5.690476, 0.00001, 1.75, 0.2],
    savefig=True,
    outdir=str(OUTDIR),
    overwrite=True,
)
fig = toi6109.plot_tql()

In [ ]:
r = toi6109.tls_results
for epoch, oc, err in zip(
    r["ttv_local_visit_epochs"], r["ttv_local_visit_oc"], r["ttv_local_visit_err"]
):
    print(f"visit around epoch {epoch:6.1f}: mean O-C = {oc * 1440:+6.1f} ± {err * 1440:.1f} min")
for png in sorted(OUTDIR.glob("TOI6109_*_ttv*.png")):
    print(png.name)
    display(Image(filename=png))

## Same runs from the command line

```bash
ql run --name TOI-216 --sectors 1,2,3,4,5,6 --exptime 120 --window-length 1 --ttv-search --save
ql run --name TOI-6109 --sectors 18,58,85 --exptime 120 --ttv-search \
  --custom-ephem 2458791.0538 0.001 5.690476 0.00001 1.75 0.2 --save
```

In the web GUI: tick **TTV search**, set Sector to `1-6` (or `18,58,85`) and Exposure Time `120`, and paste the ephemeris into **Custom Ephemeris**.